# HPM 数字化电磁算法 CAE V1.3：插件式传播后端快速复现

本笔记本演示：载入全中文工程、查看已注册传播后端、执行混合场景静态控场、对比四类后端，并输出归一化结果。

> 数值边界：仅处理波长尺度、归一化标量复场和无量纲代理指标；不输出绝对源功率、现实器件阈值、毁伤概率或作用距离。

In [1]:
from pathlib import Path
import sys

项目根目录 = Path.cwd().resolve()
if 项目根目录.name == "notebooks":
    项目根目录 = 项目根目录.parent
sys.path.insert(0, str(项目根目录 / "src"))

from hpm_platform.physics.field_backends import available_field_backends
from hpm_platform.ui.backend_explorer import run_backend_comparison, make_backend_gallery
from hpm_platform.ui.figures import make_field_figure
from hpm_platform.ui.project_model import CAEProject
from hpm_platform.ui.quick_solver import solve_project

## 1. 载入 V1.3 环境工程

In [2]:
工程 = CAEProject.load_yaml(项目根目录 / "configs" / "cae_project_v13.yaml")
print("工程名称：", 工程.meta.name)
print("传播后端：", 工程.propagation.backend)
print("启用反射面：", len(工程.active_reflectors))
print("启用孔缝：", len(工程.active_apertures))
print("启用腔体：", len(工程.active_cavities))

工程名称： HPM 数字化电磁算法 CAE V1.3 环境场景
传播后端： hybrid_scene
启用反射面： 1
启用孔缝： 1
启用腔体： 1


## 2. 查看传播后端注册表

In [3]:
for 后端 in available_field_backends():
    print(f"{后端.backend_id:24s}  {后端.display_name}  —  {后端.description}")

aperture_cavity_rom       孔缝—腔体降阶模型  —  以孔缝耦合代理和有限模态基展开描述腔体内复场，只用于快速归一化算法研究。
free_space_green          自由空间标量格林  —  仅含直达标量格林函数，适合作为算法基线与回归校验。
hybrid_scene              混合场景后端  —  直达项、一阶镜像反射与孔缝—腔体降阶分量的统一线性叠加。
image_ray                 镜像射线一阶多径  —  自由空间直达项叠加有限个平面的一阶镜像反射，适合快速多径敏感性研究。


## 3. 执行混合场景静态控场

In [4]:
结果 = solve_project(工程)
结果.metrics_frame()

,指标,数值,单位
0,传播后端,混合场景后端,
1,目标区总体 RMSE,7.4233,%
2,最差目标 RMSE,7.4701,%
3,目标区容差覆盖率,84.2809,%
4,最低目标覆盖率,81.7778,%
5,目标间 RMSE 差,0.0623,百分点
6,目标区变异系数,3.3374,%
7,区外峰值 / 目标参考,-2.2712,dB
8,区外峰值上限,-2.0,dB
9,区外峰值超限,-0.2712,dB


In [5]:
make_field_figure(结果)

## 4. 在同一工程约束下对比四类传播后端

In [6]:
对比 = run_backend_comparison(工程, fast_mode=True)
对比.records

,传播后端,后端标识,目标区RMSE/%,最低覆盖率/%,区外峰值/dB,保护区最坏超限/dB,联合判据,运行耗时/ms
0,自由空间标量格林,free_space_green,7.866052,72.444444,-2.243264,-0.415052,True,249.383171
1,镜像射线一阶多径,image_ray,7.218232,82.666667,-2.364738,0.591920,True,356.603990
2,孔缝—腔体降阶模型,aperture_cavity_rom,7.471090,75.555556,-2.135092,-0.417986,True,293.908795
3,混合场景后端,hybrid_scene,7.194087,83.111111,-2.322029,0.530219,True,394.719908


In [7]:
make_backend_gallery(对比)

## 5. 解读原则

不同后端产生的差异用于验证插件接口、模型敏感性和算法鲁棒性，不意味着某个降阶后端具有普遍的物理优越性。论文中应明确写出每种后端的适用条件、归一化方式和失效边界。